In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:

DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/3001"

import os
import cv2
import matplotlib.pyplot as plt

for folder in ["train", "valid", "test"]:
    path = os.path.join(DATA_ROOT, folder)
    print(folder, os.path.isdir(path), path)

train True /content/drive/MyDrive/Colab Notebooks/3001/train
valid True /content/drive/MyDrive/Colab Notebooks/3001/valid
test True /content/drive/MyDrive/Colab Notebooks/3001/test


In [10]:

from pathlib import Path
import cv2
import numpy as np
from skimage.segmentation import slic, mark_boundaries
from skimage.color import rgb2lab

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FOLDER = Path(
    "/content/drive/MyDrive/Colab Notebooks/3001/traintest"
)

OUTPUT_FOLDER = Path(
    "/content/drive/MyDrive/Colab Notebooks/3001/trainprocessed"
)

EDGE_FOLDER = Path(
    "/content/drive/MyDrive/Colab Notebooks/3001/trainedge"
)

# Optional: preserve original colours and draw patch boundaries
OVERLAY_FOLDER = Path(
    "/content/drive/MyDrive/Colab Notebooks/3001/trainoverlay"
)

# Canny edge detection thresholds
CANNY_LOW = 50
CANNY_HIGH = 150

# SLIC superpixel settings
N_SEGMENTS = 1000       # More = smaller patches
COMPACTNESS = 10       # Higher = more regular, compact patches
SLIC_SIGMA = 1

# K-means groups superpixels by their average colour
N_COLOURS = 6

# Overlay boundary thickness/visibility
BOUNDARY_ALPHA = 0.8

# Ignore no superpixels by default; all regions receive a colour
MIN_REGION_AREA = 1

# Repeatable cluster colours
PALETTE = np.array([
    [230,  60,  60],    # red
    [ 60, 180,  75],    # green
    [ 60, 100, 230],    # blue
    [230, 190,  40],    # yellow
    [180,  70, 200],    # purple
    [ 40, 200, 210],    # cyan
    [245, 130,  48],    # orange
    [200, 200, 200]     # grey
], dtype=np.uint8)


# ============================================================
# PROCESS ONE IMAGE
# ============================================================

def process_image(image_path, output_path, edge_path, overlay_path):

    image_bgr = cv2.imread(str(image_path))

    if image_bgr is None:
        print(f"Could not read: {image_path}")
        return False

    height, width = image_bgr.shape[:2]

    # --------------------------------------------------------
    # 1. EDGE DETECTION
    # --------------------------------------------------------

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.4)

    edges = cv2.Canny(
        blurred,
        threshold1=CANNY_LOW,
        threshold2=CANNY_HIGH
    )

    # Save the original Canny edge map
    edge_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(edge_path), edges)

    # --------------------------------------------------------
    # 2. SLIC SUPERPIXEL SEGMENTATION
    #    Segment using original appearance, not only edges.
    # --------------------------------------------------------

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    segments = slic(
        image_rgb,
        n_segments=N_SEGMENTS,
        compactness=COMPACTNESS,
        sigma=SLIC_SIGMA,
        start_label=0,
        channel_axis=-1,
        convert2lab=True
    )

    num_segments = int(segments.max()) + 1

    # --------------------------------------------------------
    # 3. CALCULATE MEAN COLOUR FOR EACH SUPERPIXEL
    #    Lab colour space is useful for perceptual colour
    #    differences.
    # --------------------------------------------------------

    lab_image = rgb2lab(image_rgb)

    features = np.zeros((num_segments, 3), dtype=np.float32)
    areas = np.bincount(segments.ravel(), minlength=num_segments)

    for channel in range(3):
        sums = np.bincount(
            segments.ravel(),
            weights=lab_image[:, :, channel].ravel(),
            minlength=num_segments
        )
        features[:, channel] = sums / np.maximum(areas, 1)

    # --------------------------------------------------------
    # 4. K-MEANS COLOUR CLUSTERING
    #    Similar-looking superpixels share a colour.
    # --------------------------------------------------------

    # Limit K if there are fewer superpixels than requested.
    k = min(N_COLOURS, num_segments)

    criteria = (
        cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
        100,
        0.2
    )

    cv2.setRNGSeed(42)

    _, labels, _ = cv2.kmeans(
        features,
        k,
        None,
        criteria,
        5,
        cv2.KMEANS_PP_CENTERS
    )

    cluster_labels = labels.flatten()

    # Assign the same colour to superpixels in the same cluster.
    palette = PALETTE[:k]
    cluster_colours = palette[cluster_labels]

    colour_patches_rgb = cluster_colours[segments]

    # --------------------------------------------------------
    # 5. OPTIONAL SMALL-REGION HANDLING
    #    Small regions remain visible, rather than turning
    #    black or disappearing.
    # --------------------------------------------------------

    # --------------------------------------------------------
    # 6. CREATE ORIGINAL-COLOUR IMAGE WITH PATCH BOUNDARIES
    # --------------------------------------------------------

    boundary_mask = np.zeros((height, width), dtype=bool)

    # Identify superpixel boundaries by comparing neighbours.
    boundary_mask[:, 1:] |= segments[:, 1:] != segments[:, :-1]
    boundary_mask[1:, :] |= segments[1:, :] != segments[:-1, :]

    # Blend dark boundaries with the original image.
    overlay_rgb = image_rgb.copy()
    boundary_colour = np.array([0, 0, 0], dtype=np.float32)

    overlay_rgb[boundary_mask] = (
        (1 - BOUNDARY_ALPHA) * image_rgb[boundary_mask].astype(np.float32)
        + BOUNDARY_ALPHA * boundary_colour
    ).astype(np.uint8)

    # --------------------------------------------------------
    # 7. SAVE OUTPUTS, PRESERVING RELATIVE FOLDER STRUCTURE
    # --------------------------------------------------------

    output_path.parent.mkdir(parents=True, exist_ok=True)
    edge_path.parent.mkdir(parents=True, exist_ok=True)
    overlay_path.parent.mkdir(parents=True, exist_ok=True)

    colour_patches_bgr = cv2.cvtColor(
        colour_patches_rgb, cv2.COLOR_RGB2BGR
    )
    overlay_bgr = cv2.cvtColor(
        overlay_rgb, cv2.COLOR_RGB2BGR
    )

    cv2.imwrite(str(output_path), colour_patches_bgr)
    cv2.imwrite(str(overlay_path), overlay_bgr)

    return True


# ============================================================
# PROCESS ALL JPG/JPEG FILES, INCLUDING SUBFOLDERS
# ============================================================

if not INPUT_FOLDER.exists():
    raise FileNotFoundError(
        f"Input folder does not exist: {INPUT_FOLDER}"
    )

image_paths = sorted(
    p for p in INPUT_FOLDER.rglob("*")
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg"}
)

print(f"Found {len(image_paths)} JPG/JPEG images.")

processed = 0
failed = 0

for index, image_path in enumerate(image_paths, start=1):

    relative_path = image_path.relative_to(INPUT_FOLDER)

    colour_output = OUTPUT_FOLDER / relative_path
    edge_output = EDGE_FOLDER / relative_path
    overlay_output = OVERLAY_FOLDER / relative_path

    try:
        success = process_image(
            image_path,
            colour_output,
            edge_output,
            overlay_output
        )

        if success:
            processed += 1
        else:
            failed += 1

    except Exception as error:
        failed += 1
        print(f"Error processing {image_path}: {error}")

    if index % 50 == 0 or index == len(image_paths):
        print(f"Progress: {index}/{len(image_paths)}")

print("\nProcessing complete.")
print(f"Successfully processed: {processed}")
print(f"Failed: {failed}")
print(f"Colour patches: {OUTPUT_FOLDER}")
print(f"Edge maps: {EDGE_FOLDER}")
print(f"Original-colour overlays: {OVERLAY_FOLDER}")


Found 10 JPG/JPEG images.
Progress: 10/10

Processing complete.
Successfully processed: 10
Failed: 0
Colour patches: /content/drive/MyDrive/Colab Notebooks/3001/trainprocessed
Edge maps: /content/drive/MyDrive/Colab Notebooks/3001/trainedge
Original-colour overlays: /content/drive/MyDrive/Colab Notebooks/3001/trainoverlay
